In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

domain = np.linspace(0, 100, 1000)

BG       = "#1a1a2e"
GRID     = "rgba(255,255,255,0.07)"
TEXT     = "#e2e8f0"
PDF_LINE = "#60a5fa"
PDF_FILL = "rgba(96,165,250,0.15)"
CDF_LINE = "#f97316"

def norm_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

def beta_pdf(x, a, b):
    t = np.clip(x / 100, 1e-10, 1 - 1e-10)
    log_pdf = (a - 1) * np.log(t) + (b - 1) * np.log(1 - t)
    pdf = np.exp(log_pdf - np.max(log_pdf))
    return pdf / 100

def uniform_pdf(x, lo, hi):
    return np.where((x >= lo) & (x <= hi), 1.0 / (hi - lo), 0.0)

def triangular_pdf(x, lo, hi, mode):
    pdf = np.where(
        x < mode,
        2 * (x - lo) / ((hi - lo) * (mode - lo)),
        2 * (hi - x) / ((hi - lo) * (hi - mode))
    )
    return np.where((x >= lo) & (x <= hi), pdf, 0.0)

def get_pdf_cdf(distribution="normal", **kwargs):
    if distribution == "normal":
        pdf = norm_pdf(domain, kwargs.get("mu", 50), kwargs.get("sigma", 12))
    elif distribution == "beta":
        pdf = beta_pdf(domain, kwargs.get("alpha", 2), kwargs.get("beta", 5))
    elif distribution == "uniform":
        pdf = uniform_pdf(domain, kwargs.get("lo", 20), kwargs.get("hi", 80))
    elif distribution == "triangular":
        pdf = triangular_pdf(domain, kwargs.get("lo", 10), kwargs.get("hi", 90), kwargs.get("mode", 50))
    else:
        raise ValueError(f"Unknown distribution: {distribution}")

    pdf = pdf / np.trapezoid(pdf, domain)
    cdf = np.zeros_like(pdf)
    cdf[1:] = np.cumsum((pdf[:-1] + pdf[1:]) / 2 * np.diff(domain))
    cdf = np.clip(cdf, 0, 1)
    cdf = (cdf - cdf.min()) / (cdf.max() - cdf.min())

    # ── Rescale to speed range [0.1, 0.9] ────────────────────────────────
    cdf = 0.1 + cdf * 0.8
    return pdf, cdf


def plot_pdf_cdf(distribution="normal", **kwargs):
    pdf, cdf = get_pdf_cdf(distribution, **kwargs)

    mean = np.trapezoid(domain * pdf, domain)
    std = np.sqrt(np.trapezoid((domain - mean) ** 2 * pdf, domain))
    median = domain[np.searchsorted(cdf, 0.5)]

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(go.Scatter(
        x=domain, y=pdf, name="PDF",
        fill="tozeroy", fillcolor=PDF_FILL,
        line=dict(color=PDF_LINE, width=2.5)
    ), secondary_y=False)

    fig.add_trace(go.Scatter(
        x=domain, y=cdf, name="Expected Speed",
        line=dict(color=CDF_LINE, width=2.5, dash="dash")
    ), secondary_y=True)

    fig.update_layout(
        title=dict(
            text=f"{distribution.capitalize()} | μ={mean:.1f}  σ={std:.1f}  median={median:.1f}",
            font=dict(color=TEXT, size=15), x=0.5
        ),
        paper_bgcolor=BG,
        plot_bgcolor=BG,
        hovermode="x unified",
        font=dict(color=TEXT),
        legend=dict(orientation="h", x=0.5, xanchor="center", y=1.08, font=dict(color=TEXT)),
    )
    fig.update_xaxes(title="x", range=[0, 100], showgrid=True,
                     gridcolor=GRID, zeroline=False, tickfont=dict(color=TEXT))
    fig.update_yaxes(title="Density", secondary_y=False,
                     showgrid=True, gridcolor=GRID, rangemode="tozero", tickfont=dict(color=TEXT))
    fig.update_yaxes(title="Expected Speed", secondary_y=True,
                     range=[0, 1.05], showgrid=False, tickfont=dict(color=TEXT))
    fig.show()

def beta_params_from_mean_std(mean, std):
    m = mean / 100
    v = (std  / 100) ** 2

    max_std = np.sqrt(m * (1 - m)) * 100
    if std >= max_std:
        raise ValueError(f"std too large for this mean. Max allowed: {max_std:.1f}")

    common = m * (1 - m) / v - 1
    alpha  = m * common
    beta   = (1 - m) * common

    return alpha, beta

In [ ]:
# ── Speed lookup: given z in [0,100], interpolate speed from CDF ──────────────
def z_to_speed(z_val, cdf):
    """Interpolate the rescaled CDF at a given z value → speed in [0.1, 0.9]"""
    return np.interp(z_val, domain, cdf)

LOG100 = np.log(1 + 100)

def pnl(research, scale, speed):
    return 200_000 * (np.log(1 + research) / LOG100) * scale * 0.07 * speed - 50_000

def pnl_surface(cdf, num_points=500):
    xi = np.linspace(0, 100, num_points)
    yi = np.linspace(0, 100, num_points)
    X, Y = np.meshgrid(xi, yi)
    Z = 100 - X - Y                     # inferred z

    # Mask infeasible: z must be in [0, 100]
    feasible = (Z >= 0) & (Z <= 100)

    speed = np.interp(Z.ravel(), domain, cdf).reshape(Z.shape)
    PNL   = np.where(feasible, pnl(X, Y, speed), np.nan)

    return xi, yi, PNL, Z

def find_optimal(cdf, num_points=500):
    xi, yi, PNL, _ = pnl_surface(cdf, num_points)
    idx = np.nanargmax(PNL)
    i, j = np.unravel_index(idx, PNL.shape)
    opt_x, opt_y = xi[j], yi[i]
    opt_z     = 100 - opt_x - opt_y
    opt_speed = np.interp(opt_z, domain, cdf)
    opt_pnl   = np.nanmax(PNL)
    return opt_x, opt_y, opt_z, opt_speed, opt_pnl

def plot_pnl_surface(distribution, num_points=500, **kwargs):
    _, cdf = get_pdf_cdf(distribution, **kwargs)
    xi, yi, PNL, Z = pnl_surface(cdf, num_points)
    opt_x, opt_y, opt_z, opt_speed, opt_pnl = find_optimal(cdf, num_points)

    fig = go.Figure()

    fig.add_trace(go.Surface(
        x=xi, y=yi, z=PNL,
        colorscale="Viridis",
        colorbar=dict(
            title=dict(text="PnL ($)", font=dict(color="#e2e8f0")),
            tickprefix="$",
            tickfont=dict(color="#e2e8f0")
        ),
        hovertemplate=(
            "Research: %{x:.1f}<br>"
            "Scale: %{y:.1f}<br>"
            "Speed: %{customdata:.1f}<br>"
            "PnL: $%{z:,.0f}<extra></extra>"
        ),
        customdata=Z[:, :, np.newaxis],

    ))

    # Optimal point marker
    fig.add_trace(go.Scatter3d(
        x=[opt_x], y=[opt_y], z=[opt_pnl],
        mode="markers+text",
        marker=dict(size=7, color="#f43f5e", symbol="diamond"),
        text=[f"Research={opt_x:.1f}, Scale={opt_y:.1f}<br>Speed={opt_z:.1f}, Speed Value={opt_speed:.3f}"],
        textfont=dict(color="#f43f5e", size=11),
        textposition="top center",
        name="Optimal",
        showlegend=False,
    ))

    zero_plane = np.zeros_like(PNL)
    fig.add_trace(go.Surface(
        x=xi, y=yi, z=zero_plane,
        colorscale=[[0, "rgba(255,50,50,0.30)"], [1, "rgba(255,50,50,0.30)"]],
        showscale=False,
        hoverinfo="skip",
        name="PnL = 0",
    ))

    BG   = "#1a1a2e"
    GRID = "rgba(255,255,255,0.07)"
    TEXT = "#e2e8f0"

    fig.update_layout(
        title=dict(
            text=(f"PnL surface  |  "
                f"Distrubution: {distribution}, "
                f"Optimal: x={opt_x:.1f}, y={opt_y:.1f}, z={opt_z:.1f}, "
                f"speed={opt_speed:.3f}  →  PnL=${opt_pnl:,.0f}"),
            font=dict(color=TEXT, size=12), x=0.5
        ),
        scene=dict(
            xaxis=dict(title=dict(text="Research Investment (0–100)", font=dict(color=TEXT)),
                    backgroundcolor=BG, gridcolor=GRID, tickfont=dict(color=TEXT)),
            yaxis=dict(title=dict(text="Scale Investment (0–100)", font=dict(color=TEXT)),
                    backgroundcolor=BG, gridcolor=GRID, tickfont=dict(color=TEXT)),
            zaxis=dict(title=dict(text="PnL ($)",   font=dict(color=TEXT)),
                    backgroundcolor=BG, gridcolor=GRID, tickfont=dict(color=TEXT),
                    tickprefix="$"),
            bgcolor=BG,
        ),
        paper_bgcolor=BG,
        font=dict(color=TEXT),
        margin=dict(t=60, b=20, l=20, r=20),
        height=650,
    )

    fig.show()
    print(f"\nOptimal allocation:")
    print(f"  Research = {opt_x:.2f}  (out of 100)")
    print(f"  Scale = {opt_y:.2f}  (out of 100)")
    print(f"  Speed Investment = {opt_z:.2f}  (inferred, determines speed)")
    print(f"  Expected Speed Value = {opt_speed:.4f}  (from CDF of beta distribution at z)")
    print(f"  PnL   = ${opt_pnl:,.2f}")


mean = 20
std = 15
alpha, beta = beta_params_from_mean_std(mean, std)

plot_pdf_cdf("uniform", lo=0, hi=100)
plot_pnl_surface("uniform", lo=0, hi=100)

plot_pdf_cdf("normal", mu=mean, sigma=std)
plot_pnl_surface("normal", mu=mean, sigma=std)

plot_pdf_cdf("beta", alpha=alpha, beta=beta)
plot_pnl_surface("beta", alpha=alpha, beta=beta)

plot_pdf_cdf("triangular", lo=10, hi=90, mode=40)
plot_pnl_surface("triangular", lo=10, hi=90, mode=40)

In [30]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def analyze_top_percentile(cdf, num_points=500, percentile=95):
    """
    Find all (research, scale, speed_inv) combos in the top percentile of PnL,
    then for each unique speed investment level, return the best research/scale pair.
    """
    xi = np.linspace(0, 100, num_points)
    yi = np.linspace(0, 100, num_points)
    X, Y = np.meshgrid(xi, yi)
    Z = 100 - X - Y

    feasible = (Z >= 0) & (Z <= 100)
    speed = np.interp(Z.ravel(), domain, cdf).reshape(Z.shape)
    PNL = np.where(feasible, pnl(X, Y, speed), np.nan)

    # ── Top-percentile mask ───────────────────────────────────────────────────
    threshold = np.nanpercentile(PNL, percentile)
    top_mask  = feasible & (PNL >= threshold)

    top_research  = X[top_mask]
    top_scale     = Y[top_mask]
    top_speed_inv = Z[top_mask]
    top_pnl       = PNL[top_mask]
    top_speed_val = speed[top_mask]

    # ── For each speed-investment bucket, find the best research/scale ────────
    # Round speed_inv to nearest integer for bucketing
    speed_inv_rounded = np.round(top_speed_inv).astype(int)
    unique_z = np.unique(speed_inv_rounded)

    best_by_speed = []
    for z_val in unique_z:
        mask = speed_inv_rounded == z_val
        idx  = np.argmax(top_pnl[mask])

        best_by_speed.append({
            "speed_inv":   z_val,
            "speed_val":   top_speed_val[mask][idx],
            "research":    top_research[mask][idx],
            "scale":       top_scale[mask][idx],
            "pnl":         top_pnl[mask][idx],
        })

    return {
        "threshold":    threshold,
        "top_research":  top_research,
        "top_scale":     top_scale,
        "top_speed_inv": top_speed_inv,
        "top_pnl":       top_pnl,
        "best_by_speed": best_by_speed,
        "full_PNL":      PNL,
        "xi": xi, "yi": yi,
        "X": X,   "Y": Y, "Z": Z,
    }


def plot_top_percentile(distribution, num_points=500, percentile=95, **kwargs):
    _, cdf = get_pdf_cdf(distribution, **kwargs)
    res    = analyze_top_percentile(cdf, num_points, percentile)

    best     = res["best_by_speed"]
    z_vals   = [b["speed_inv"] for b in best]
    pnl_vals = [b["pnl"]       for b in best]

    BG, GRID, TEXT = "#1a1a2e", "rgba(255,255,255,0.07)", "#e2e8f0"

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=z_vals, y=pnl_vals,
        mode="lines+markers",
        line=dict(color="#7c3aed", width=2),
        marker=dict(size=5, color="#f43f5e"),
        hovertemplate="SpeedInv=%{x}<br>PnL=$%{y:,.0f}<extra></extra>",
    ))
    fig.add_shape(
        type="line",
        x0=min(z_vals), x1=max(z_vals),
        y0=res["threshold"], y1=res["threshold"],
        line=dict(color="rgba(255,255,255,0.3)", dash="dot", width=1),
    )
    fig.add_annotation(
        x=max(z_vals), y=res["threshold"],
        text=f"P{percentile}: ${res['threshold']:,.0f}",
        showarrow=False, font=dict(color=TEXT, size=10),
        xanchor="right", yanchor="bottom",
    )

    fig.update_layout(
        title=dict(
            text=f"Top {100-percentile}% PnL | Distribution: {distribution} | Threshold: ${res['threshold']:,.0f}"
                  f"| mean={kwargs.get('mu', "")}, std={kwargs.get('sigma', "")}",
            font=dict(color=TEXT, size=13), x=0.5,
        ),
        xaxis=dict(title="Speed investment (z)", gridcolor=GRID, tickfont=dict(color=TEXT), title_font=dict(color=TEXT)),
        yaxis=dict(title="Best PnL ($)", tickprefix="$", gridcolor=GRID, tickfont=dict(color=TEXT), title_font=dict(color=TEXT)),
        paper_bgcolor=BG, plot_bgcolor=BG,
        font=dict(color=TEXT),
        showlegend=False,
        height=400,
        margin=dict(t=80, b=40, l=40, r=40),
    )

    fig.show()

# ── Run ───────────────────────────────────────────────────────────────────────
mean  = 35
std   = 15
alpha, beta_p = beta_params_from_mean_std(mean, std)

means = [20, 25, 30, 35, 40]
stds = [15, 20, 25]

plot_pdf_cdf("uniform", lo=0, hi=100)
plot_top_percentile("uniform", lo=0, hi=100)

for mean in means:
    for std in stds:
        plot_pdf_cdf("normal", mu=mean, sigma=std)
        plot_top_percentile("normal",     mu=mean, sigma=std)